In [ ]:
import os

DATASET = "/kaggle/input"

for root, dirs, files in os.walk(DATASET):
    tiffs = [f for f in files if f.lower().endswith(".tiff")]

    if len(tiffs) > 0:
        print("\nFolder:", root)
        print("TIFF Count:", len(tiffs))
        print("First 10 Files:")
        for f in sorted(tiffs)[:10]:
            print("   ", f)
        print("-"*80)

In [ ]:
%pip install -q segmentation-models-pytorch rasterio tifffile opencv-python scikit-learn tqdm

In [ ]:
import os
import glob
import cv2
import random
import rasterio
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
thermal_files = sorted(
    glob.glob(
        "/kaggle/input/**/*.TIFF",
        recursive=True
    )
)

thermal_files = [
    f for f in thermal_files
    if "geo_thermal_tiff_celsius" in f
]

print("Total Thermal Images :", len(thermal_files))
print(thermal_files[0])

In [ ]:
UNET_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "unet_fire_segmentation_final.pth":
            UNET_PATH = os.path.join(root, file)

print("UNET_PATH")
print(UNET_PATH)

In [ ]:
unet = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
)

In [ ]:
checkpoint = torch.load(
    UNET_PATH,
    map_location=device
)

unet.load_state_dict(
    checkpoint["model_state_dict"]
)

threshold = checkpoint["threshold"]

unet = unet.to(device)
unet.eval()

print("Loaded Successfully")
print("Threshold :", threshold)

In [ ]:
def preprocess(path):

    with rasterio.open(path) as src:
        img = src.read(1)

    img = np.nan_to_num(img)

    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    img = (img * 255).astype(np.uint8)

    img = cv2.resize(img, (256,256))

    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    tensor = torch.from_numpy(
        rgb.astype(np.float32)/255.0
    ).permute(2,0,1).unsqueeze(0)

    return img, tensor

In [ ]:
fire_masks = []

for path in tqdm(thermal_files):

    image, tensor = preprocess(path)

    tensor = tensor.to(device)

    with torch.no_grad():

        pred = unet(tensor)

        pred = torch.sigmoid(pred)

        mask = (pred > threshold).float()

    fire_masks.append(mask.squeeze().cpu().numpy())

print("Generated Masks :", len(fire_masks))

In [ ]:
idx = random.randint(0, len(fire_masks)-1)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(preprocess(thermal_files[idx])[0], cmap="inferno")
plt.title("Thermal")

plt.subplot(1,2,2)
plt.imshow(fire_masks[idx], cmap="gray")
plt.title("Generated Fire Mask")

plt.tight_layout()
plt.show()

In [ ]:
from collections import defaultdict
import os

sequence_groups = defaultdict(list)

for path, mask in zip(thermal_files, fire_masks):

    folder = os.path.dirname(path)

    sequence_groups[folder].append(mask)

print("Total Sequences:", len(sequence_groups))

for folder in sequence_groups:
    print(folder, len(sequence_groups[folder]))

In [ ]:
SEQUENCE_LENGTH = 5

X = []
Y = []
SAMPLE_GROUPS = []

for folder in sequence_groups:

    masks = sequence_groups[folder]

    for i in range(len(masks) - SEQUENCE_LENGTH):

        X.append(masks[i:i + SEQUENCE_LENGTH])
        Y.append(masks[i + SEQUENCE_LENGTH])
        SAMPLE_GROUPS.append(folder)

X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.float32)
SAMPLE_GROUPS = np.array(SAMPLE_GROUPS)

print("Input sequences:", X.shape)
print("Targets:", Y.shape)
print("Sequence groups:", len(np.unique(SAMPLE_GROUPS)))

In [ ]:
class FireSequenceDataset(Dataset):

    def __init__(self, X, Y, augment=False):
        if len(X) != len(Y):
            raise ValueError("X and Y must contain the same number of samples")
        self.X = X
        self.Y = Y
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        sequence = torch.as_tensor(self.X[idx], dtype=torch.float32)
        target = torch.as_tensor(self.Y[idx], dtype=torch.float32)

        if self.augment:
            if torch.rand(()) < 0.5:
                sequence = torch.flip(sequence, dims=(-1,))
                target = torch.flip(target, dims=(-1,))

            if torch.rand(()) < 0.5:
                sequence = torch.flip(sequence, dims=(-2,))
                target = torch.flip(target, dims=(-2,))

        return sequence.unsqueeze(1), target.unsqueeze(0)

In [ ]:
unique_groups = np.unique(SAMPLE_GROUPS)

if len(unique_groups) < 2:
    raise ValueError(
        "At least two sequence folders are required for a group-based train/validation split"
    )

rng = np.random.default_rng(42)
shuffled_groups = rng.permutation(unique_groups)
validation_group_count = max(1, int(round(len(shuffled_groups) * 0.2)))
validation_groups = shuffled_groups[:validation_group_count]
training_groups = shuffled_groups[validation_group_count:]

train_indices = np.flatnonzero(np.isin(SAMPLE_GROUPS, training_groups))
val_indices = np.flatnonzero(np.isin(SAMPLE_GROUPS, validation_groups))

if len(train_indices) == 0 or len(val_indices) == 0:
    raise ValueError(
        "The grouped split produced an empty training or validation set; add more sequence folders."
    )

X_train, X_val = X[train_indices], X[val_indices]
Y_train, Y_val = Y[train_indices], Y[val_indices]

train_dataset = FireSequenceDataset(
    X_train,
    Y_train,
    augment=True
)

val_dataset = FireSequenceDataset(
    X_val,
    Y_val,
    augment=False
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Train groups:", len(np.unique(SAMPLE_GROUPS[train_indices])))
print("Validation groups:", len(np.unique(SAMPLE_GROUPS[val_indices])))

In [ ]:
# num_workers=0 keeps this notebook portable across Windows and notebook kernels.
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Train Batches:", len(train_loader))
print("Validation Batches:", len(val_loader))

In [ ]:
# Cell 16

sequence, target = next(iter(train_loader))

print("Sequence Shape :", sequence.shape)
print("Target Shape   :", target.shape)

In [ ]:
# Cell 17

sample_sequence = sequence[0]

plt.figure(figsize=(15,3))

for i in range(SEQUENCE_LENGTH):

    plt.subplot(1, SEQUENCE_LENGTH, i + 1)

    plt.imshow(sample_sequence[i,0], cmap="gray")

    plt.title(f"Frame {i+1}")

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# Cell 18

plt.figure(figsize=(5,5))

plt.imshow(target[0,0], cmap="gray")

plt.title("Target Frame")

plt.axis("off")

plt.show()

In [ ]:
# Cell 19

print("Sequence Length :", sample_sequence.shape[0])

print("Frames Used")

for i in range(SEQUENCE_LENGTH):
    print("Frame", i+1)

print("Target = Next Frame")

In [ ]:
# Cell 20

import gc

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory Cleared")

In [ ]:
from collections import defaultdict
import os

groups = defaultdict(list)

for path in thermal_files:
    folder = os.path.dirname(path)
    groups[folder].append(path)

print("Total sequences:", len(groups))

for folder, files in list(groups.items())[:10]:
    print(folder)
    print("Frames:", len(files))
    print("First:", os.path.basename(sorted(files)[0]))
    print("Last :", os.path.basename(sorted(files)[-1]))
    print("-" * 60)

In [ ]:
class ConvLSTMCell(nn.Module):

    def __init__(self, input_channels, hidden_channels, kernel_size=3):

        super().__init__()

        padding = kernel_size // 2

        self.hidden_channels = hidden_channels

        self.conv = nn.Conv2d(
            input_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size,
            padding=padding
        )

    def forward(self, x, h, c):

        combined = torch.cat([x, h], dim=1)

        gates = self.conv(combined)

        i, f, o, g = torch.chunk(gates, 4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c = f * c + i * g
        h = o * torch.tanh(c)

        return h, c

In [ ]:
class ConvLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.hidden_channels = 32

        self.cell = ConvLSTMCell(
            input_channels=1,
            hidden_channels=self.hidden_channels
        )

        self.output = nn.Conv2d(
            self.hidden_channels,
            1,
            kernel_size=1
        )

    def forward(self, x):

        batch, seq_len, ch, h, w = x.size()

        hidden = torch.zeros(
            batch,
            self.hidden_channels,
            h,
            w,
            device=x.device
        )

        cell = torch.zeros_like(hidden)

        for t in range(seq_len):

            hidden, cell = self.cell(
                x[:, t],
                hidden,
                cell
            )

        out = self.output(hidden)

        return out

In [ ]:
model = ConvLSTM().to(device)

print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
def dice_score(pred, target):

    pred = torch.sigmoid(pred)

    pred = (pred > 0.5).float()

    intersection = (pred * target).sum()

    return (
        2 * intersection + 1e-8
    ) / (
        pred.sum() +
        target.sum() +
        1e-8
    )

In [ ]:
def train_epoch():

    model.train()

    total_loss = 0

    total_dice = 0

    for sequence, target in train_loader:

        sequence = sequence.to(device)

        target = target.to(device)

        optimizer.zero_grad()

        output = model(sequence)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        total_dice += dice_score(
            output,
            target
        ).item()

    return (
        total_loss / len(train_loader),
        total_dice / len(train_loader)
    )

In [ ]:
def validate():

    model.eval()

    total_loss = 0

    total_dice = 0

    with torch.no_grad():

        for sequence, target in val_loader:

            sequence = sequence.to(device)

            target = target.to(device)

            output = model(sequence)

            loss = criterion(output, target)

            total_loss += loss.item()

            total_dice += dice_score(
                output,
                target
            ).item()

    return (
        total_loss / len(val_loader),
        total_dice / len(val_loader)
    )

In [ ]:
EPOCHS = 20

best_dice = 0

history = []

for epoch in range(EPOCHS):

    train_loss, train_dice = train_epoch()

    val_loss, val_dice = validate()

    history.append([
        train_loss,
        val_loss,
        train_dice,
        val_dice
    ])

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Val Loss   : {val_loss:.4f}"
    )

    print(
        f"Train Dice : {train_dice:.4f}"
    )

    print(
        f"Val Dice   : {val_dice:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice

        torch.save(
            model.state_dict(),
            "future_fire_prediction_augmented_best.pth"
        )

        print("Best Model Saved")

In [ ]:
history = np.array(history)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history[:,0], label="Train")
plt.plot(history[:,1], label="Validation")
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history[:,2], label="Train")
plt.plot(history[:,3], label="Validation")
plt.title("Dice")
plt.legend()

plt.show()

In [ ]:
import os

print("Saved:", os.path.exists("future_fire_prediction.pth"))

In [ ]:
model.eval()

sequence, target = next(iter(val_loader))

sequence = sequence.to(device)
target = target.to(device)

with torch.no_grad():
    prediction = model(sequence)
    prediction = torch.sigmoid(prediction)
    prediction = (prediction > 0.5).float()

current = sequence[0, -1, 0].cpu().numpy()
ground_truth = target[0, 0].cpu().numpy()
predicted = prediction[0, 0].cpu().numpy()

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(current, cmap="gray")
plt.title("Last Input Frame")

plt.subplot(1,3,2)
plt.imshow(ground_truth, cmap="gray")
plt.title("Ground Truth Next Frame")

plt.subplot(1,3,3)
plt.imshow(predicted, cmap="gray")
plt.title("Predicted Next Frame")

plt.tight_layout()
plt.show()

In [ ]:
difference = np.abs(predicted - ground_truth)

print("Different Pixels:", np.sum(difference))

plt.figure(figsize=(6,6))
plt.imshow(difference, cmap="hot")
plt.title("Prediction Error")
plt.colorbar()
plt.show()

In [ ]:
torch.save(model.state_dict(), "future_fire_prediction_augmented_best.pth")

print("Saved Successfully")

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "sequence_length": SEQUENCE_LENGTH,
        "hidden_channels": 32,
        "val_dice": best_dice,
        "epochs": EPOCHS
    },
    "future_fire_prediction_augmented_best.pth"
)

print("Model Saved Successfully")